# WM 2026 – CV Block: Trikot-Klassifikation

Dieses Notebook trainiert einen ViT-basierten Bildklassifikator, der anhand eines Trikotfotos die Nation erkennt.

**Pipeline:**  
**`CV (Trikot → Nation)`** → `ML (Nation → WM-Prognose)` → `NLP (Prognose → Erklärung)`

Basiert auf demselben Ansatz wie Übung 2 (Pokemon-Klassifikation), angewendet auf 16 WM-Nationen.

---

## Setup

In [ ]:
import subprocess, sys

# Pakete direkt in der aktiven Kernel-Umgebung installieren
packages = [
    "accelerate>=1.1.0",
    "transformers",
    "datasets",
    "evaluate",
    "torch",
    "torchvision",
    "huggingface_hub",
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

# Verifizieren
import accelerate
print(f"Python:     {sys.executable}")
print(f"accelerate: {accelerate.__version__}")
print("Alle Pakete bereit ✓")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import (
    ViTForImageClassification,
    ViTImageProcessor,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset as HFDataset
import evaluate
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

DATA_DIR   = Path('../../data/jerseys')
MODEL_DIR  = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

HF_MODEL_NAME = 'tschool01/wm2026-jersey-classifier'

NATIONS = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f'{len(NATIONS)} Nationen: {NATIONS}')

## 1. Daten-Check & EDA

In [ ]:
# Bilder pro Nation zählen
counts = {}
for nation in NATIONS:
    imgs = list((DATA_DIR / nation).glob('*.jpg')) + \
           list((DATA_DIR / nation).glob('*.jpeg')) + \
           list((DATA_DIR / nation).glob('*.png')) + \
           list((DATA_DIR / nation).glob('*.webp'))
    counts[nation] = len(imgs)

print('Bilder pro Nation:')
for k, v in sorted(counts.items()):
    bar = '█' * v
    print(f'  {k:15s}: {v:3d}  {bar}')

print(f'\nGesamt: {sum(counts.values())} Bilder')

if any(v == 0 for v in counts.values()):
    print('\n⚠️  Leere Ordner gefunden! Bilder sammeln bevor Training startet.')

In [ ]:
# Beispielbilder anzeigen (je 1 pro Nation)
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
axes = axes.flatten()

for i, nation in enumerate(NATIONS):
    imgs = list((DATA_DIR / nation).glob('*.*'))
    if imgs:
        img = Image.open(imgs[0]).convert('RGB')
        axes[i].imshow(img)
        axes[i].set_title(nation, fontsize=8)
    axes[i].axis('off')

plt.suptitle('Beispielbilder – je 1 Trikot pro Nation', y=1.02)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'eda_samples.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. Daten vorbereiten

**Preprocessing:** Resize auf 224×224, Normalisierung nach ImageNet-Standard.  
**Augmentation:** Random Horizontal Flip, Colour Jitter (nur Training).

In [ ]:
label2id = {n: i for i, n in enumerate(NATIONS)}
id2label = {i: n for n, i in label2id.items()}
print('Label-Mapping:', label2id)

# Alle Bildpfade + Labels einlesen
all_paths, all_labels = [], []
for nation in NATIONS:
    imgs = list((DATA_DIR / nation).glob('*.jpg')) + \
           list((DATA_DIR / nation).glob('*.jpeg')) + \
           list((DATA_DIR / nation).glob('*.png')) + \
           list((DATA_DIR / nation).glob('*.webp'))
    for p in imgs:
        all_paths.append(str(p))
        all_labels.append(label2id[nation])

print(f'Gesamt: {len(all_paths)} Bilder')

In [ ]:
from sklearn.model_selection import train_test_split

# 70% Train / 15% Val / 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(
    all_paths, all_labels, test_size=0.3, stratify=all_labels, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

In [ ]:
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

class JerseyDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        pixel_values = self.transform(img)
        return {'pixel_values': pixel_values, 'labels': self.labels[idx]}

train_ds = JerseyDataset(X_train, y_train, train_transforms)
val_ds   = JerseyDataset(X_val,   y_val,   val_transforms)
test_ds  = JerseyDataset(X_test,  y_test,  val_transforms)

print(f'Datasets bereit: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')

## 3. Modell – Iteration 1: ViT Transfer Learning (Baseline)

**Modell:** `google/vit-base-patch16-224-in21k` (vortrainiert auf ImageNet-21k)  
**Strategie:** Nur den Klassifikations-Head neu trainieren (frozen backbone)

In [ ]:
def load_vit(freeze_backbone=True):
    model = ViTForImageClassification.from_pretrained(
        'google/vit-base-patch16-224-in21k',
        num_labels=len(NATIONS),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    if freeze_backbone:
        for name, param in model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Trainierbare Parameter: {trainable:,} / {total:,}')
    return model

model_iter1 = load_vit(freeze_backbone=True)

In [ ]:
accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

training_args_iter1 = TrainingArguments(
    output_dir=str(MODEL_DIR / 'vit-iter1'),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=10,
    report_to='none',
)

trainer_iter1 = Trainer(
    model=model_iter1,
    args=training_args_iter1,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Training Iteration 1 startet...')
trainer_iter1.train()

In [ ]:
# Iteration 1 – Evaluation auf Val-Set
results_iter1 = trainer_iter1.evaluate()
print('Iteration 1 Val-Accuracy:', results_iter1['eval_accuracy'])

## 4. Modell – Iteration 2: Fine-Tuning (Full Backbone)

**Änderungen gegenüber Iter. 1:**
- Gesamter Backbone wird trainiert (kein Freeze)
- Niedrigere Learning Rate (1e-5)
- Mehr Augmentation

In [ ]:
model_iter2 = load_vit(freeze_backbone=False)

training_args_iter2 = TrainingArguments(
    output_dir=str(MODEL_DIR / 'vit-iter2'),
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=10,
    report_to='none',
)

trainer_iter2 = Trainer(
    model=model_iter2,
    args=training_args_iter2,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Training Iteration 2 startet...')
trainer_iter2.train()

In [ ]:
results_iter2 = trainer_iter2.evaluate()
print('Iteration 2 Val-Accuracy:', results_iter2['eval_accuracy'])

print('\nVergleich:')
print(f'  Iter 1 (frozen):     {results_iter1["eval_accuracy"]:.3f}')
print(f'  Iter 2 (full FT):    {results_iter2["eval_accuracy"]:.3f}')

## 5. Evaluation auf Test-Set

In [ ]:
# Bestes Modell auf Test-Set evaluieren
best_trainer = trainer_iter2  # ggf. trainer_iter1 falls Iter 1 besser

test_results = best_trainer.predict(test_ds)
preds = np.argmax(test_results.predictions, axis=-1)
true  = test_results.label_ids

print('Test-Set Accuracy:', (preds == true).mean())
print()
print(classification_report(true, preds, target_names=NATIONS))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(true, preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=NATIONS, yticklabels=NATIONS)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix – Trikot-Klassifikation')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'cv_confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Modell auf Hugging Face hochladen

In [ ]:
import getpass
hf_token = getpass.getpass("Hugging Face Token eingeben: ")
print("Token gespeichert ✓")

In [ ]:
from huggingface_hub import HfApi

# Modell hochladen
best_trainer.model.push_to_hub(HF_MODEL_NAME, token=hf_token)

# Processor hochladen
processor.push_to_hub(HF_MODEL_NAME, token=hf_token)

print(f"✓ Modell hochgeladen: https://huggingface.co/{HF_MODEL_NAME}")

## 7. Inference-Funktion (für App)

Diese Funktion wird von `app.py` importiert. Output: erkannte Nation → Input für ML-Block.

In [ ]:
from transformers import pipeline as hf_pipeline
from openai import OpenAI
import base64, json

# Eigenes trainiertes Modell
vit_classifier = hf_pipeline(
    'image-classification',
    model=HF_MODEL_NAME
)

# CLIP zero-shot (wie in Übung 2)
CLIP_LABELS = [f'{n} football jersey national team' for n in NATIONS]
clip_classifier = hf_pipeline(
    'zero-shot-image-classification',
    model='openai/clip-vit-large-patch14'
)

def classify_vit(image_path):
    results = vit_classifier(image_path)
    return {r['label']: round(r['score'], 4) for r in results}

def classify_clip(image_path):
    results = clip_classifier(image_path, candidate_labels=NATIONS)
    return {r['label']: round(r['score'], 4) for r in results}

def classify_openai(image_path, api_key):
    client = OpenAI(api_key=api_key)
    with open(image_path, 'rb') as f:
        img_b64 = base64.b64encode(f.read()).decode('utf-8')
    ext = image_path.split('.')[-1].lower()
    mime = 'image/png' if ext == 'png' else 'image/jpeg'

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'text', 'text': (
                    f'Classify this football jersey image as exactly one of these nations: {', '.join(NATIONS)}. '
                    'Reply with a JSON object where keys are nation names and values are confidence scores '
                    'between 0 and 1 that sum to 1. Output only the JSON, nothing else.'
                )},
                {'type': 'image_url', 'image_url': {'url': f'data:{mime};base64,{img_b64}'}},
            ]
        }],
        max_tokens=300,
    )
    text = response.choices[0].message.content.strip()
    if '```' in text:
        text = text.split('```')[1]
        if text.startswith('json'): text = text[4:]
    try:
        return {k: round(float(v), 4) for k, v in json.loads(text).items()}
    except Exception:
        return {'error': text}

def get_top_nation(image_path):
    """Gibt die erkannte Nation zurück (Top-1 vom eigenen ViT-Modell)."""
    results = classify_vit(image_path)
    return max(results, key=results.get)

# Test
# print(get_top_nation('../../data/jerseys/brazil/test.jpg'))

## 8. Zusammenfassung

| Iteration | Ziel | Änderungen | Modell | Val-Accuracy | Änderung |
|---|---|---|---|---|---|
| 1 | Baseline Transfer Learning | Backbone frozen, nur Head trainiert | ViT-base (ImageNet-21k) | 15.0% | – |
| 2 | Full Fine-Tuning | Gesamter Backbone trainiert, LR 1e-5, Warmup | ViT-base (ImageNet-21k) | 31.7% | +16.7% vs Iter. 1 |

**Test-Set Accuracy:** 38.3% (16 Klassen, Zufallslevel = 6.25%)

**Limitation:** Mit nur 25 Bildern pro Nation ist die Accuracy erwartungsgemäss beschränkt. CLIP und GPT-4o werden als stärkere Alternativen verglichen.

**Finales Modell:** ViT-base (Iter. 2), hochgeladen als `tschool01/wm2026-jersey-classifier`  
**Output → ML Block:** `get_top_nation(image)` gibt den Ländernamen zurück (Input für `predict_nation()`)

## 9. Modellvergleich – Quantitative Evaluation (CLIP vs. GPT-4o vs. ViT)

In [ ]:
# CLIP auf demselben Test-Set evaluieren (X_test / y_test aus Zelle 9)
correct_clip = 0
for img_path, true_label in zip(X_test, y_test):
    results = clip_classifier(img_path, candidate_labels=NATIONS)
    pred_nation = results[0]['label']
    if label2id[pred_nation] == true_label:
        correct_clip += 1

clip_accuracy = correct_clip / len(X_test)
print(f"CLIP Top-1 Accuracy auf Test-Set (N={len(X_test)}): {clip_accuracy:.1%} ({correct_clip}/{len(X_test)})")
print(f"ViT  Top-1 Accuracy auf Test-Set (N={len(X_test)}): 38.3% (23/60)")
print(f"→ CLIP ist {'besser' if clip_accuracy > 0.383 else 'schlechter oder gleich'} als das eigene ViT-Modell")

In [ ]:
# GPT-4o auf 1 Bild pro Nation aus dem Test-Set (= 16 API-Calls)
import getpass
openai_key = getpass.getpass("OpenAI API Key: ")

FOLDER_TO_PROPER = {
    "argentina": "Argentina", "australia": "Australia", "belgium": "Belgium",
    "brazil": "Brazil", "colombia": "Colombia", "croatia": "Croatia",
    "england": "England", "france": "France", "germany": "Germany",
    "japan": "Japan", "netherlands": "Netherlands", "portugal": "Portugal",
    "southafrica": "South Africa", "spain": "Spain", "switzerland": "Switzerland",
    "usa": "USA"
}
TARGET_NATIONS_PROPER = list(FOLDER_TO_PROPER.values())

nation_test_img = {}
for img_path, true_label in zip(X_test, y_test):
    nation = id2label[true_label]
    if nation not in nation_test_img:
        nation_test_img[nation] = img_path

from openai import OpenAI
client_eval = OpenAI(api_key=openai_key)

correct_gpt, total_gpt = 0, 0
for nation_folder, img_path in sorted(nation_test_img.items()):
    true_proper = FOLDER_TO_PROPER[nation_folder]
    with open(img_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = client_eval.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": [
            {"type": "text", "text": (
                "Which national football team jersey is this? "
                "Answer with exactly one of: " + ", ".join(TARGET_NATIONS_PROPER) + ". "
                "Reply only the nation name, nothing else."
            )},
            {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + b64}}
        ]}],
        max_tokens=20
    )
    pred = resp.choices[0].message.content.strip()
    correct = (pred == true_proper)
    correct_gpt += int(correct)
    total_gpt += 1
    mark = "V" if correct else "X"
    print(true_proper.ljust(15) + ": pred=" + pred.ljust(15) + " " + mark)

gpt_accuracy = correct_gpt / total_gpt
print("\nGPT-4o Top-1 Accuracy (1 Bild/Nation, N=" + str(total_gpt) + "): " + f"{gpt_accuracy:.1%}")
print()
print("=== Modellvergleich Zusammenfassung ===")
print("  ViT   (fine-tuned, 60 Bilder): 38.3%")
print("  CLIP  (zero-shot,  60 Bilder): " + f"{clip_accuracy:.1%}")
print("  GPT-4o (1 Bild/Nation,  16):   " + f"{gpt_accuracy:.1%}")
